In [ ]:
# Databricks notebook source
# 00_check_source_changes
# Checks whether any source file changed since the last successful pipeline run.

from datetime import datetime, timezone

from pyspark.sql import functions as F

dbutils.widgets.dropdown("force_run", "false", ["true", "false"])
force_run = dbutils.widgets.get("force_run").lower() == "true"

CATALOG = "decide_catalog"
SCHEMA = "decide_schema"
VOLUME_PATH = "/Volumes/decide_catalog/decide_schema/decide_volume"
STATE_TABLE = f"{CATALOG}.{SCHEMA}.source_file_state"

SOURCE_FILES = [
    "DECIDE_COMPIL_DATA__202501272012.xlsx",
    "MTA_DECIDE_Ugent2025.xlsx",
    "DECIDE_MTA_UGENT_BAC_AERO_14nov2022.xlsx",
    "DECIDE_MTA_UGENTBAC_MYCO_14nov2022.xlsx",
    "250808_data_RGD_DECIDE.xlsx",
    "Jade_2021_Final_Anonymised_data_Only_2023-04-20.v2.xlsx",
    "Jade_2022_Final_Anonymised_data_Only_2023-04-21.xlsx",
    "Final_Anonymised_data_Only_2023_2025-03-04.xlsx",
    "AllBovineRespiratory_NegativesIncluded.csv",
    "DECIDE_final_version.xlsx",
    "gd_labresults.csv",
]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

listed_files = {item.name: item for item in dbutils.fs.ls(VOLUME_PATH)}
missing_files = [name for name in SOURCE_FILES if name not in listed_files]
if missing_files:
    raise FileNotFoundError("Missing expected source files in volume: " + ", ".join(missing_files))

checked_at = datetime.now(timezone.utc)
current_rows = [
    {
        "file_name": name,
        "file_path": listed_files[name].path,
        "file_size": int(listed_files[name].size),
        "modification_time_ms": int(listed_files[name].modificationTime),
        "checked_at_utc": checked_at,
    }
    for name in SOURCE_FILES
]
current_df = spark.createDataFrame(current_rows)

if spark.catalog.tableExists(STATE_TABLE):
    previous_df = spark.table(STATE_TABLE).select(
        "file_name",
        F.col("file_size").alias("previous_file_size"),
        F.col("modification_time_ms").alias("previous_modification_time_ms"),
    )
else:
    previous_df = spark.createDataFrame([], "file_name string, previous_file_size long, previous_modification_time_ms long")

comparison_df = (
    current_df.join(previous_df, on="file_name", how="left")
    .withColumn(
        "changed",
        F.col("previous_file_size").isNull()
        | (F.col("file_size") != F.col("previous_file_size"))
        | (F.col("modification_time_ms") != F.col("previous_modification_time_ms")),
    )
)

changed_files = [row.file_name for row in comparison_df.filter("changed").select("file_name").collect()]
should_run = force_run or len(changed_files) > 0

dbutils.jobs.taskValues.set(key="should_run", value=str(should_run).lower())
dbutils.jobs.taskValues.set(key="changed_files", value=",".join(changed_files))

print(f"force_run={force_run}")
print(f"should_run={should_run}")
print("changed_files=" + (", ".join(changed_files) if changed_files else "None"))

# Do not update STATE_TABLE here. The union notebook updates it only after all cleaning tasks complete successfully.
dbutils.notebook.exit("CHANGED" if should_run else "NO_CHANGES")
